Importation des bibliothèques nécessaires, ensuite on fait la connexion à Elasticsearch et on vérifie le fonctionnement de la connexion

In [34]:
from elasticsearch import Elasticsearch
import json

# Connexion à Elasticsearch
es = Elasticsearch("http://elasticsearch:9200")

# Vérifier que la connexion fonctionne
print(es.info())

{'name': '8024d90a7fda', 'cluster_name': 'docker-cluster', 'cluster_uuid': '5cA-zghnRGeBZOCAYOedow', 'version': {'number': '8.10.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6d20dd8ce62365be9b1aca96427de4622e970e9e', 'build_date': '2023-09-19T08:16:24.564900370Z', 'build_snapshot': False, 'lucene_version': '9.7.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


Requête 1 : Compter le nombre de films dans movies_clean, et c'est une preuve que l'ingestion des données a bien fonctionné

In [35]:
result = es.count(index="movies_clean")
print(f"Nombre total de films : {result['count']}")

Nombre total de films : 662083


Requête 2 : Récupérer un film pour voir l'inspection des données 

In [36]:
result = es.search(index="movies_clean", body={"size": 1})
film = result['hits']['hits'][0]['_source']
print(json.dumps(film, indent=2, ensure_ascii=False))

{
  "@version": "1",
  "message": "869432,Lake Forest Park,Drama,en,A coming-of-age film about a group of outsider teenagers in Seattle dealing with the consequences of accidental gun violence in their hometown.,1.4,,2021-09-18,0.0,0.0,60.0,Released,,0.0,0.0,Alex Hofstrand-Luka Huffines-Tres Huffines-Dylan Lehmann-Kesiah Manival-Clara McHale-Ribot-Jack Paradise-Lily Rand-Weihong Zheng,seattle usa-high school-grief-gun violence-washington state-woman director-washington,/9O5MOGlwMFVd0zHEwF0R0xrgXYk.jpg,,",
  "id": "869432",
  "credits": "Alex Hofstrand-Luka Huffines-Tres Huffines-Dylan Lehmann-Kesiah Manival-Clara McHale-Ribot-Jack Paradise-Lily Rand-Weihong Zheng",
  "production_companies": null,
  "event": {
    "original": "869432,Lake Forest Park,Drama,en,A coming-of-age film about a group of outsider teenagers in Seattle dealing with the consequences of accidental gun violence in their hometown.,1.4,,2021-09-18,0.0,0.0,60.0,Released,,0.0,0.0,Alex Hofstrand-Luka Huffines-Tres Huffin

Requête 3 : Recherche full text sur le film batman (le titre du film), et grace à notre analyzer qu'on a définit dans le mapping il arrive à récupérer par exemple tous les batman, BATMAN, BAT MAN, Btaman

In [37]:
result = es.search(index="movies_clean", body={
    "query": {
        "match": {
            "title": "batman"
        }
    }
})
print(f"Films trouvés : {result['hits']['total']['value']}")
for hit in result['hits']['hits']:
    print(f"- {hit['_source'].get('title', 'Sans titre')}")

Films trouvés : 113
- Batman
- The Batman
- Batman
- Batman
- What, Batman? King, Batman!!
- The Batman: Making-Of
- Batman: Hush
- Batman Ninja
- Batman Begins
- Batman Forever


Requête 4 : Affichage du top 10 des films les mieux notés, poue cela on utilise un filtre range pour garder uniquement les films avec une note >=  8/10, et on trie par note décroissante pour avoir les meilleurs en premier.

In [38]:
result = es.search(index="movies_clean", body={
    "query": {
        "range": {
            "vote_average": {"gte": 8.0}
        }
    },
    "sort": [{"vote_average": "desc"}],
    "size": 10
})
print(f"TOP 10 films les mieux notés: ")
for hit in result['hits']['hits']:
    src = hit['_source']
    print(f"- {src.get('title')} | Note: {src.get('vote_average')}")

TOP 10 films les mieux notés: 
- Kathryn Upside Down | Note: 10.0
- Morazán | Note: 10.0
- Kids in Brick Houses | Note: 10.0
- Chelsea FC - Season Review 2002/03 | Note: 10.0
- Isaac Asimov's Robots | Note: 10.0
- Pintakasi | Note: 10.0
- Ringgo: The Dog Shooter | Note: 10.0
- Hello, Baby! | Note: 10.0
- On the Road | Note: 10.0
- Yt | Note: 10.0


Requête 5 : Affichage des films d'action en anglais les mieux notés avec une note >= 7, pour cela on utilise le must qui est obligatoire et change le score pour récupérer les films d'action, et filter qui est obligatoire mais change pas le score pour la langue en anglais et la note >= 7

In [39]:
result = es.search(index="movies_clean", body={
    "query": {
        "bool": {
            "must": [
                {"match": {"genres": "Action"}}
            ],
            "filter": [
                {"term": {"original_language": "en"}},
                {"range": {"vote_average": {"gte": 7.0}}}
            ]
        }
    }
})
print(f"Films d'action en anglais les mieux notés : ")
for hit in result['hits']['hits'][:5]:
    src = hit['_source']
    print(f"- {src.get('title')} | Note: {src.get('vote_average')}")

Films d'action en anglais les mieux notés : 
- Kurt Angle: The Essential Collection | Note: 10.0
- Bellator 207: Mitrione vs. Bader | Note: 10.0
- WCW Halloween Havoc 1999 | Note: 10.0
- ROH Best In The World 2018 | Note: 10.0
- UFC 204: Bisping vs. Henderson 2 | Note: 7.0


Requête 6 : Affichage des films populaires sortis après 2010 dont leurs durée >= 80min, pour cela on utilise une requête bool pour combiner les 3 conditions

In [40]:
result = es.search(index="movies_clean", body={
    "query": {
        "bool": {
            "must": [
                {"range": {"popularity": {"gte": 50}}}
            ],
            "filter": [
                {"range": {"release_date": {"gte": "2010-01-01"}}},
                {"range": {"runtime": {"gte": 80}}}
            ]
        }
    },
    "sort": [{"popularity": "desc"}],
    "size": 10
})
print(f"Films populaires sortis après 2010 : ")
for hit in result['hits']['hits']:
    src = hit['_source']
    print(f"- {src.get('title')} | Popularité: {src.get('popularity')}")

Films populaires sortis après 2010 : 
- The Northman | Popularité: 3669.153
- Fantastic Beasts: The Secrets of Dumbledore | Popularité: 3456.961
- The Lost City | Popularité: 3190.602
- Morbius | Popularité: 3124.049
- Sonic the Hedgehog 2 | Popularité: 2984.257
- A Day to Die | Popularité: 2847.136
- Uncharted | Popularité: 2735.373
- Memory | Popularité: 2366.025
- Jurassic World Dominion | Popularité: 2240.933
- Doctor Strange in the Multiverse of Madness | Popularité: 2080.956


Requête 7 : Affichage des films flops avec un gros budget >= 100M$ et une note <= 5 et moins de 100 votes 

In [41]:
result = es.search(index="movies_clean", body={
    "query": {
        "bool": {
            "must": [
                {"range": {"budget": {"gte": 100000000}}}
            ],
            "filter": [
                {"range": {"vote_average": {"lte": 5.0}}},
                {"range": {"vote_count": {"gte": 100}}}
            ]
        }
    },
    "sort": [{"budget": "desc"}]
})
print(f"Grands films flops : ")
for hit in result['hits']['hits'][:5]:
    src = hit['_source']
    print(f"- {src.get('title')} | Budget: {src.get('budget')}$ | Note: {src.get('vote_average')}")

Grands films flops : 
- Speed 2: Cruise Control | Budget: 160000000$ | Note: 4.5
- The Last Airbender | Budget: 150000000$ | Note: 4.7
- Batman & Robin | Budget: 125000000$ | Note: 4.3
- Fantastic Four | Budget: 120000000$ | Note: 4.4
- Catwoman | Budget: 100000000$ | Note: 4.6


Requête 8 : Affichage des bons films dont la langue n'est pas en français ni en anglais, pour cela on utilise le must_not pour excule facilment les films qui sont en anglais et français

In [42]:
result = es.search(index="movies_clean", body={
    "query": {
        "bool": {
            "must": [
                {"range": {"vote_average": {"gte": 7.5}}}
            ],
            "must_not": [
                {"term": {"original_language": "en"}},
                {"term": {"original_language": "fr"}}
            ],
            "filter": [
                {"range": {"vote_count": {"gte": 200}}}
            ]
        }
    },
    "sort": [{"vote_average": "desc"}],
    "size": 10
})
print(f"Films monde bien notés : ")
for hit in result['hits']['hits']:
    src = hit['_source']
    print(f"- {src.get('title')} | Langue: {src.get('original_language')} | Note: {src.get('vote_average')}")

Films monde bien notés : 
- BTS World Tour: Love Yourself - Japan Edition | Langue: ko | Note: 9.2
- Dilwale Dulhania Le Jayenge | Langue: hi | Note: 8.7
- Impossible Things | Langue: es | Note: 8.7
- Burn the Stage: The Movie | Langue: ko | Note: 8.6
- Your Eyes Tell | Langue: ja | Note: 8.6
- Dou kyu sei – Classmates | Langue: ja | Note: 8.6
- Bring the Soul: The Movie | Langue: ko | Note: 8.6
- Seven Samurai | Langue: ja | Note: 8.5
- Violet Evergarden: The Movie | Langue: ja | Note: 8.5
- Josee, the Tiger and the Fish | Langue: ja | Note: 8.5


Requête 9 : Rechecrhe du mot "love" dans les titres, description et tagline des films, plus le mot "love" est dans plusieurs champs sur un film plus son score augmente, pour cela on utilise le should au moins un champ doit correspandre au mot "love"

In [43]:
result = es.search(index="movies_clean", body={
    "query": {
        "bool": {
            "should": [
                {"match": {"title": "love"}},
                {"match": {"overview": "love"}},
                {"match": {"tagline": "love"}}
            ],
            "minimum_should_match": 1,
            "filter": [
                {"range": {"vote_average": {"gte": 6.0}}}
            ]
        }
    }
})
print(f"Films avec 'love' : {result['hits']['total']['value']}")
for hit in result['hits']['hits'][:5]:
    src = hit['_source']
    print(f"- {src.get('title')} | Note: {src.get('vote_average')}")

Films avec 'love' : 10000
- Liberating Love | Note: 10.0
- Loving | Note: 6.7
- Love | Note: 7.1
- Love and... | Note: 7.9
- Love Actually | Note: 7.1


Requête 10 : Affichage de combien de films par langue ? On va juste voir les statistiques non les films, pour cela on fait une aggrégation et on mets size = 0 pour avoir que les statistiques

In [44]:
result = es.search(index="movies_clean", body={
    "size": 0,
    "aggs": {
        "films_par_langue": {
            "terms": {
                "field": "original_language",
                "size": 10
            }
        }
    }
})
print("Top 10 des langues :")
for bucket in result['aggregations']['films_par_langue']['buckets']:
    print(f"- {bucket['key']} : {bucket['doc_count']} films")

Top 10 des langues :
- en : 350160 films
- fr : 41261 films
- es : 37440 films
- de : 34646 films
- ja : 24958 films
- pt : 19176 films
- ru : 16231 films
- it : 15748 films
- zh : 15577 films
- ko : 8754 films


Requête 11 : Affichage de la note moyenne des films par langue, pour cela on fait une aggrégation imbriquée on fait les groupes de films par langue puis on calcule leurs moyennes

In [45]:
result = es.search(index="movies_clean", body={
    "size": 0,
    "aggs": {
        "par_langue": {
            "terms": {
                "field": "original_language",
                "size": 10
            },
            "aggs": {
                "note_moyenne": {
                    "avg": {"field": "vote_average"}
                }
            }
        }
    }
})
print("Note moyenne par langue :")
for bucket in result['aggregations']['par_langue']['buckets']:
    print(f"- {bucket['key']} : {bucket['note_moyenne']['value']:.2f}/10")

Note moyenne par langue :
- en : 2.26/10
- fr : 2.55/10
- es : 2.76/10
- de : 1.96/10
- ja : 2.32/10
- pt : 1.84/10
- ru : 2.13/10
- it : 3.00/10
- zh : 1.49/10
- ko : 2.41/10


Requête 12 : Affichage de films produits par année, on utilise une agrégation date_histogram qui regroupe les films par période de temps. On filtre sur des dates entre 1900 et 2026 car certains films ont des dates aberrantes dans le dataset comme 2090 et 2099, qui est une anomalie détectée lors du nettoyage des données.

In [46]:
result = es.search(index="movies_clean", body={
    "size": 0,
    "query": {
        "range": {
            "release_date": {
                "gte": "1900-01-01",
                "lte": "2026-12-31"
            }
        }
    },
    "aggs": {
        "films_par_annee": {
            "date_histogram": {
                "field": "release_date",
                "calendar_interval": "year"
            }
        }
    }
})

print("Production par année (10 dernières années réelles) :")
buckets = result['aggregations']['films_par_annee']['buckets']

# Filtrer les buckets avec au moins 1 film
buckets_valides = [b for b in buckets if b['doc_count'] > 0]

# Afficher les 10 dernières années
for bucket in buckets_valides[-10:]:
    annee = bucket['key_as_string'][:4]
    count = bucket['doc_count']
    print(f"- {annee} : {count} films")

Production par année (10 dernières années réelles) :
- 2017 : 25825 films
- 2018 : 27217 films
- 2019 : 29947 films
- 2020 : 29596 films
- 2021 : 29598 films
- 2022 : 9378 films
- 2023 : 220 films
- 2024 : 22 films
- 2025 : 6 films
- 2026 : 3 films
